# Cumulative Chemical Discovery Analysis (Full Corpus)

This notebook connects to the Offline LanceDB and visualizes the rate of discovery of unique chemicals across the entire 266M+ record corpus.


In [ ]:
import lancedb
import matplotlib.pyplot as plt
import json
import pandas as pd
import pyarrow as pa
from tqdm import tqdm
import time
import os

# 1. Connect to LanceDB
DB_PATH = "/data/chen/GNPS_SIM/pmc_lancedb_index"
TABLE_NAME = "papers_index"

try:
    db = lancedb.connect(DB_PATH)
    tbl = db.open_table(TABLE_NAME)
    print(f"Connected to {DB_PATH}, Table: {TABLE_NAME}")
    total_rows = tbl.count_rows()
    print(f"Total Rows in DB: {total_rows}")
except Exception as e:
    print(f"Connection Failed: {e}")
    tbl = None


In [ ]:
# 2. Stream Full Dataset
# WARNING: This will take hours for 266M rows.

BATCH_SIZE = 10000
report_every = 1000000 # Report progress every 1M rows

cumulative_counts = []
doc_counts = []
seen_chemicals = set()
processed_count = 0
ner_valid_count = 0
start_time = time.time()

if tbl:
    print("Starting Full Scan...")
    
    # Select only ner_results column for speed
    # Use tbl.to_lance() for better streaming performance if available, or tbl.search()
    scanner = tbl.search().select(["ner_results"]).to_arrow()
    batches = scanner.to_batches(max_chunksize=BATCH_SIZE)
    
    for batch in batches:
        # Convert to python objects is slow, try to work with string column if possible
        # Batch.column(0) is ner_results
        
        # Arrow to pylist is reasonably fast for strings
        ner_strings = batch.column(0).to_pylist()
        
        for ner_str in ner_strings:
            processed_count += 1
            
            # Fast rejection of empty objects
            if not ner_str or len(ner_str) <= 4: # "{}" is len 2, "null" is len 4
                continue
            
            # If it looks like valid JSON
            try:
                # Only parse if it looks like it has content (naive check)
                if ':' not in ner_str:
                    continue
                    
                # data = json.loads(ner_str) # Still expensive
                # Optimization: Simple Regex or String Find?
                # No, correct parsing needed for keys.
                
                data = json.loads(ner_str)
                if not isinstance(data, dict) or not data:
                    continue
                    
                ner_valid_count += 1
                seen_chemicals.update(data.keys())
                
                # Sampling for plot (don't save every single point, save every 1000th valid doc)
                if ner_valid_count % 1000 == 0:
                    cumulative_counts.append(len(seen_chemicals))
                    doc_counts.append(ner_valid_count)
                    
            except Exception:
                continue
        
        if processed_count % report_every == 0:
            elapsed = time.time() - start_time
            rate = processed_count / elapsed
            print(f"Processed {processed_count} rows. Valid NER: {ner_valid_count}. Unique Chems: {len(seen_chemicals)}. Rate: {rate:.0f} rows/s")
            
    print(f"Finished Full Scan.")
    print(f"Total Rows: {processed_count}")
    print(f"Valid NER Docs: {ner_valid_count}")
    print(f"Total Unique Chemicals: {len(seen_chemicals)}")

In [ ]:
# 3. Plotting
if cumulative_counts:
    plt.figure(figsize=(14, 7))
    
    plt.plot(doc_counts, cumulative_counts, color='darkgreen', linewidth=2, label='Cumulative Unique Chemicals')
    
    plt.xlabel("Valid Articles/Chunks Processed")
    plt.ylabel("Cumulative Count of Unique Chemicals")
    plt.title("Discovery Rate of Unique Chemicals (Full 266M Corpus Scan)")
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.legend()
    plt.tight_layout()
    plt.savefig("ner_discovery_full.png")
    plt.show()
    
    # Also save CSV data
    pd.DataFrame({'valid_docs': doc_counts, 'unique_chemicals': cumulative_counts}).to_csv("ner_discovery_full_data.csv", index=False)
else:
    print("No data to plot.")